# Verify BOS breakout v2 changes

This notebook runs small, synthetic tests to sanity-check:

- ATR-based stop loss (mode `"atr"`)
- Percent-of-equity risk sizing
- Partial exit at 1R and ATR-based trailing stop

It does **not** replace full backtests, but helps quickly confirm the wiring and basic math.


In [1]:
import os
import sys
from dataclasses import asdict

# Make project code importable when running from this notebook
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Also add the QuantConnect-style project folder ("code_"), so that
# top-level packages like `entry_exit_rules`, `risk_management`, etc. work
CODE_ROOT = os.path.join(PROJECT_ROOT, "code_")
if os.path.isdir(CODE_ROOT) and CODE_ROOT not in sys.path:
    sys.path.append(CODE_ROOT)

from entry_exit_rules.entry_exit import (
    Bar,
    SwingLevels,
    PositionDirection,
    TakeProfitMode,
    SameBarSlTpRule,
    BosSignal,
    TradePlan,
    detect_bos_signal,
    plan_trade_from_signal,
    check_exit_rules,
    check_partial_1r_reached,
    compute_trailing_stop,
)
from stop_loss.stop_loss import StopLossManager
from risk_management.risk import RiskConfig, size_position
from atr_module.atr_module import get_atr_from_bars

print("Imports OK. Project root:", PROJECT_ROOT)


Imports OK. Project root: /Users/timkuz/Documents/GitHub/breakout-strategy-v2


In [5]:
# --- Helper: build simple synthetic series and a BOS signal ---

import math


def build_synthetic_bos_long_series(n: int = 40):
    """Build a simple uptrend with a BOS long on the penultimate bar.

    We keep things deterministic and small; no regime or swing detection here.
    """
    bars: list[Bar] = []
    price = 100.0
    for i in range(n):
        # simple stair-step up move
        o = price
        h = o + 1.0
        l = o - 1.0
        c = o + 0.5
        bars.append(Bar(open=o, high=h, low=l, close=c))
        price = c

    # Define last swing low/high manually for this synthetic test
    swing_levels = SwingLevels(
        last_swing_high_price=bars[-3].close,  # BOS up when close[-2] > this
        last_swing_low_price=bars[-6].low,
    )

    signal_index = len(bars) - 2  # t
    bos_signal = BosSignal(direction=PositionDirection.LONG, signal_candle_index=signal_index)
    return bars, swing_levels, bos_signal


bars, swing_levels, bos_signal = build_synthetic_bos_long_series()
print("Bars:", len(bars))
print("Swing levels:", swing_levels)
print("BOS signal:", bos_signal)


Bars: 40
Swing levels: SwingLevels(last_swing_high_price=119.0, last_swing_low_price=116.0)
BOS signal: BosSignal(direction=<PositionDirection.LONG: 'LONG'>, signal_candle_index=38)


In [6]:
# --- Test 1: ATR-based SL + percent-of-equity sizing ---

EQUITY = 100_000.0
risk_cfg = RiskConfig(
    risk_pct=0.01,            # risk 1% of equity per trade
    max_position_size=None,
    min_stop_distance=None,
    max_leverage=3.0,
    use_buying_power_cap=False,
)

sl_manager = StopLossManager(mode="atr", k_sl=2.0, atr_period=14)

entry_candle_index = bos_signal.signal_candle_index + 1
entry_price = bars[entry_candle_index].open

plan = plan_trade_from_signal(
    bars=bars,
    bos_signal=bos_signal,
    swing_levels=swing_levels,
    stop_loss_manager=sl_manager,
    tp_mode=TakeProfitMode.RR_BASED,
    tp_mult=3.0,
    risk_config=risk_cfg,
    equity=EQUITY,
    buying_power_cash=EQUITY,  # generous for this test
    position_sizer=size_position,
)

print("Trade plan (dict):")
print(asdict(plan))

# Quick sanity checks
R = abs(plan.entry_price - plan.sl_price)
print("R (entry - SL):", R)
print("Expect LONG SL < entry:", plan.sl_price < plan.entry_price)
print("RR-TP distance / R:", (plan.tp_price - plan.entry_price) / R)


Trade plan (dict):
{'direction': <PositionDirection.LONG: 'LONG'>, 'signal_candle_index': 38, 'entry_candle_index': 39, 'entry_price': 119.5, 'sl_price': 115.5, 'tp_price': 131.5, 'quantity': 250.0}
R (entry - SL): 4.0
Expect LONG SL < entry: True
RR-TP distance / R: 3.0


In [7]:
# --- Test 2: clean partial exit at 1R and ATR trailing ---

# We use the TradePlan from Test 1 (already printed):
# entry_price = 119.5, sl_price = 115.5, tp_price = 131.5, direction = LONG

entry = plan.entry_price
sl = plan.sl_price
initial_tp = plan.tp_price

print("Test 2 TradePlan:")
print(" entry=", entry, " sl=", sl, " tp=", initial_tp, " direction=", plan.direction)

# 1R for LONG
R = entry - sl  # 119.5 - 115.5 = 4.0
one_r_target = entry + R
print("Computed R:", R)
print("1R target (entry + R):", one_r_target)

# Clean 1R bar: touches 1R, does NOT hit SL, does NOT hit full TP
bar_clean_1r = Bar(
    open=122.0,
    high=124.0,   # >= 123.5 (1R target)
    low=121.5,    # > 115.5 (no SL)
    close=123.0,  # < 131.5 (no full TP)
)

print("Clean 1R bar:", bar_clean_1r)

# 1) Check that RR-based SL/TP logic alone does NOT exit on this bar
exit_event_clean = check_exit_rules(
    bar=bar_clean_1r,
    direction=plan.direction,
    sl_price=sl,
    tp_price=initial_tp,
    same_bar_rule=SameBarSlTpRule.WORST_CASE,
)
print("check_exit_rules (clean 1R bar):", exit_event_clean)

# 2) Check that 1R condition IS detected
hit_1r_clean = check_partial_1r_reached(
    bar=bar_clean_1r,
    direction=plan.direction,
    entry_price=entry,
    sl_price=sl,
    r_mult=1.0,
)
print("check_partial_1r_reached (clean 1R bar):", hit_1r_clean)

print("Clean 1R sanity:")
print("  SL touched?", bar_clean_1r.low <= sl)
print("  TP touched?", bar_clean_1r.high >= initial_tp)

# 3) After partial exit, emulate ATR-based trailing stop using this bar
bars_with_clean_1r = bars + [bar_clean_1r]
atr_at_trailing_bar = get_atr_from_bars(
    bars_with_clean_1r,
    len(bars_with_clean_1r) - 1,
    sl_manager.atr_period,
)

trail_stop = compute_trailing_stop(
    close=bar_clean_1r.close,
    atr=atr_at_trailing_bar,
    direction=plan.direction,
    old_stop=sl,
    k_trail=2.0,
)

print("ATR at trailing bar (clean 1R):", atr_at_trailing_bar)
print("Old SL:", sl)
print("New trailing stop:", trail_stop)
print("Trailing stop > old SL?", trail_stop > sl)
print("Trailing stop <= close?", trail_stop <= bar_clean_1r.close)


Test 2 TradePlan:
 entry= 119.5  sl= 115.5  tp= 131.5  direction= PositionDirection.LONG
Computed R: 4.0
1R target (entry + R): 123.5
Clean 1R bar: Bar(open=122.0, high=124.0, low=121.5, close=123.0, volume=None, time=None)
check_exit_rules (clean 1R bar): None
check_partial_1r_reached (clean 1R bar): True
Clean 1R sanity:
  SL touched? False
  TP touched? False
ATR at trailing bar (clean 1R): 2.142857142857143
Old SL: 115.5
New trailing stop: 118.71428571428571
Trailing stop > old SL? True
Trailing stop <= close? True


In [8]:
# --- Test 3: ambiguous same-bar 1R + SL case ---

# Now construct a bar where price both reaches 1R *and* trades through SL.

entry = plan.entry_price
sl = plan.sl_price
initial_tp = plan.tp_price
R = entry - sl
one_r_target = entry + R

bar_ambig = Bar(
    open=entry,
    high=one_r_target + 0.5,  # reaches above 1R
    low=sl - 0.5,             # trades below SL
    close=entry,              # arbitrary close
)

print("Ambiguous bar (1R + SL in same bar):", bar_ambig)

exit_event_ambig = check_exit_rules(
    bar=bar_ambig,
    direction=plan.direction,
    sl_price=sl,
    tp_price=initial_tp,
    same_bar_rule=SameBarSlTpRule.WORST_CASE,
)

hit_1r_ambig = check_partial_1r_reached(
    bar=bar_ambig,
    direction=plan.direction,
    entry_price=entry,
    sl_price=sl,
    r_mult=1.0,
)

print("check_exit_rules (ambiguous bar):", exit_event_ambig)
print("check_partial_1r_reached (ambiguous bar):", hit_1r_ambig)

print("Ambiguous sanity:")
print("  SL touched?", bar_ambig.low <= sl)
print("  1R touched?", bar_ambig.high >= one_r_target)
print("  TP touched?", bar_ambig.high >= initial_tp)

print("Priority behavior explanation:")
print("  With V1 exit rules (SameBarSlTpRule.WORST_CASE), SL takes precedence over TP.")
print("  Partial exit at 1R is layered on top of these rules in the strategy,")
print("  so in a real run a pure SL bar would still be treated as a full stop-out.")


Ambiguous bar (1R + SL in same bar): Bar(open=119.5, high=124.0, low=115.0, close=119.5, volume=None, time=None)
check_exit_rules (ambiguous bar): TradeExit(exit_price=115.5, exit_reason=<ExitReason.SL: 'SL'>)
check_partial_1r_reached (ambiguous bar): True
Ambiguous sanity:
  SL touched? True
  1R touched? True
  TP touched? False
Priority behavior explanation:
  With V1 exit rules (SameBarSlTpRule.WORST_CASE), SL takes precedence over TP.
  Partial exit at 1R is layered on top of these rules in the strategy,
  so in a real run a pure SL bar would still be treated as a full stop-out.


In [9]:
# --- Test 4: FULL TRADE LIFECYCLE (LONG, partial + trailing) ---

print("=== Test 4: FULL TRADE LIFECYCLE (LONG) ===")

# Use the TradePlan from Test 1
entry = plan.entry_price
sl_initial = plan.sl_price
tp = plan.tp_price
initial_qty = float(plan.quantity)

R = entry - sl_initial
one_r_target = entry + R

print("Initial trade state:")
print("  direction        =", plan.direction)
print("  entry_price      =", entry)
print("  initial_sl       =", sl_initial)
print("  tp_price         =", tp)
print("  initial_qty      =", initial_qty)
print("  R                =", R)
print("  target_1R        =", one_r_target)

# Local state machine (simplified version of main.py logic)
remaining_qty = initial_qty
partial_exit_done = False
trailing_active = False
trailing_stop = sl_initial
num_trailing_updates = 0
trade_open = True
partial_closed_qty = 0.0
final_exit_qty = 0.0
final_exit_reason = None

# Start from existing bars (uptrend from Test 1) and append lifecycle bars
bars_lifecycle = list(bars)

# Synthetic bars after entry_candle_index (Bar A .. E)
# Conditions:
#  - Bar A: no SL, no 1R
#  - Bar B: clean 1R (partial exit), no SL, no full TP
#  - Bar C/D: move higher, trailing stop steps up
#  - Bar E: pullback into trailing stop (exit remaining)

bar_A = Bar(open=120.0, high=122.0, low=119.0, close=121.0)
bar_B = Bar(open=122.0, high=124.0, low=121.5, close=123.0)   # clean 1R bar
bar_C = Bar(open=123.2, high=126.0, low=122.8, close=125.0)
bar_D = Bar(open=125.0, high=127.0, low=124.2, close=126.2)
bar_E = Bar(open=126.0, high=126.4, low=120.0, close=121.0)   # deep pullback designed to hit trailing

bars_scenario = [("Bar A", bar_A), ("Bar B", bar_B), ("Bar C", bar_C), ("Bar D", bar_D), ("Bar E", bar_E)]

for name, bar in bars_scenario:
    print("\n---", name, "---")
    bars_lifecycle.append(bar)
    idx = len(bars_lifecycle) - 1

    print("Bar:", bar)

    if not trade_open:
        print("  Trade already closed; skipping further logic.")
        continue

    # 1) If partial not done yet: check for hard SL/TP first, then partial 1R
    if not partial_exit_done:
        exit_event = check_exit_rules(
            bar=bar,
            direction=plan.direction,
            sl_price=sl_initial,
            tp_price=tp,
            same_bar_rule=SameBarSlTpRule.WORST_CASE,
        )
        hit_1r = check_partial_1r_reached(
            bar=bar,
            direction=plan.direction,
            entry_price=entry,
            sl_price=sl_initial,
            r_mult=1.0,
        )

        print("  check_exit_rules      =", exit_event)
        print("  check_partial_1r      =", hit_1r)
        print("  partial_exit_done     =", partial_exit_done)
        print("  remaining_qty before  =", remaining_qty)

        if exit_event is not None:
            # In this lifecycle test we design bars so that A-D do not trigger SL/TP.
            trade_open = False
            final_exit_qty = remaining_qty
            remaining_qty = 0.0
            final_exit_reason = exit_event.exit_reason.value
            print("  HARD EXIT (unexpected in this scenario):", exit_event)
            continue

        if hit_1r:
            # Execute partial exit at 50% of current position
            partial_qty = remaining_qty * 0.5
            remaining_qty -= partial_qty
            partial_closed_qty += partial_qty
            partial_exit_done = True
            trailing_active = True
            # Trailing is activated, but SL itself is not moved yet on this bar
            trailing_stop = sl_initial

            print("  >>> PARTIAL EXIT TRIGGERED @1R")
            print("  closed_qty           =", partial_qty)
            print("  remaining_qty after  =", remaining_qty)
            print("  partial_exit_done    =", partial_exit_done)
            print("  trailing_active      =", trailing_active)
            print("  trailing_stop        =", trailing_stop)
        else:
            print("  No exit on this bar; trade remains fully open.")

    else:
        # 2) After partial: update trailing stop then check for trailing exit
        #    This mimics main.py: trailing updates on bar close only.
        atr_here = get_atr_from_bars(bars_lifecycle, idx, sl_manager.atr_period)
        old_trail = trailing_stop
        trailing_stop = compute_trailing_stop(
            close=bar.close,
            atr=atr_here,
            direction=plan.direction,
            old_stop=trailing_stop,
            k_trail=2.0,
        )
        if trailing_stop != old_trail:
            num_trailing_updates += 1

        # Use a far TP so only the trailing SL can realistically hit
        far_tp = entry + 1e9 if plan.direction == PositionDirection.LONG else 0.0
        exit_event = check_exit_rules(
            bar=bar,
            direction=plan.direction,
            sl_price=trailing_stop,
            tp_price=far_tp,
            same_bar_rule=SameBarSlTpRule.WORST_CASE,
        )

        print("  ATR at bar           =", atr_here)
        print("  old_trailing_stop    =", old_trail)
        print("  new_trailing_stop    =", trailing_stop)
        print("  trailing_moved_up?   =", trailing_stop >= old_trail)
        print("  trailing <= close?   =", trailing_stop <= bar.close)
        print("  check_exit_rules     =", exit_event)

        # Explicit check that the bar actually touches the trailing stop level
        touched_trailing = bar.low <= trailing_stop if plan.direction == PositionDirection.LONG else bar.high >= trailing_stop
        print("  trailing_stop        =", trailing_stop)
        print("  bar.low              =", bar.low)
        print("  trailing_touched?    =", touched_trailing)
        print("  remaining_qty before =", remaining_qty)

        if exit_event is not None:
            # In our scenario we expect the final bar to act like a trailing stop hit
            trade_open = False
            final_exit_qty = remaining_qty
            remaining_qty = 0.0
            final_exit_reason = "TRAILING_STOP"
            print("  >>> TRAILING EXIT TRIGGERED (via check_exit_rules)")
            print("  exit_reason          =", final_exit_reason)
            print("  exit_qty             =", final_exit_qty)
        elif touched_trailing:
            # Fallback: explicitly close by trailing stop if price touched the level
            trade_open = False
            final_exit_qty = remaining_qty
            remaining_qty = 0.0
            final_exit_reason = "TRAILING_STOP"
            print("  >>> TRAILING EXIT TRIGGERED (explicit trailing touch)")
            print("  exit_reason          =", final_exit_reason)
            print("  exit_qty             =", final_exit_qty)
        else:
            print("  No trailing exit on this bar; trade remains open.")
            print("  remaining_qty        =", remaining_qty)

# --- Summary ---
print("\n=== Test 4 SUMMARY ===")
print("initial_qty             =", initial_qty)
print("partial_qty_closed      =", partial_closed_qty)
print("remaining_qty_after_partial =", remaining_qty if partial_exit_done else initial_qty)
print("final_exit_qty          =", final_exit_qty)
print("partial_exit_done       =", partial_exit_done)
print("trailing_was_activated  =", trailing_active)
print("num_trailing_updates    =", num_trailing_updates)
print("final_exit_reason       =", final_exit_reason)
print("trade_open              =", trade_open)
print("remaining_qty           =", remaining_qty)

lifecycle_test_passed = (
    partial_exit_done
    and abs(partial_closed_qty - initial_qty * 0.5) < 1e-9
    and final_exit_qty == initial_qty * 0.5
    and not trade_open
    and trailing_active
    and num_trailing_updates >= 1
    and final_exit_reason == "TRAILING_STOP"
    and remaining_qty == 0.0
)

print("lifecycle_test_passed   =", lifecycle_test_passed)


=== Test 4: FULL TRADE LIFECYCLE (LONG) ===
Initial trade state:
  direction        = PositionDirection.LONG
  entry_price      = 119.5
  initial_sl       = 115.5
  tp_price         = 131.5
  initial_qty      = 250.0
  R                = 4.0
  target_1R        = 123.5

--- Bar A ---
Bar: Bar(open=120.0, high=122.0, low=119.0, close=121.0, volume=None, time=None)
  check_exit_rules      = None
  check_partial_1r      = False
  partial_exit_done     = False
  remaining_qty before  = 250.0
  No exit on this bar; trade remains fully open.

--- Bar B ---
Bar: Bar(open=122.0, high=124.0, low=121.5, close=123.0, volume=None, time=None)
  check_exit_rules      = None
  check_partial_1r      = True
  partial_exit_done     = False
  remaining_qty before  = 250.0
  >>> PARTIAL EXIT TRIGGERED @1R
  closed_qty           = 125.0
  remaining_qty after  = 125.0
  partial_exit_done    = True
  trailing_active      = True
  trailing_stop        = 115.5

--- Bar C ---
Bar: Bar(open=123.2, high=126.0, low